# Downstream Task Examples

This notebook demonstrates two downstream tasks using pre-trained BulkRNABERT representations:

1. **Cancer type classification** — predict cancer cohort (5 TCGA cohorts) from RNA-seq
2. **Pan-cancer survival** — predict relative survival risk from RNA-seq

Both models share the same BulkRNABERT encoder backbone. The survival model outputs a **log partial hazard** (linear predictor in Cox regression): a higher value means higher predicted risk (shorter expected survival). Absolute values are not interpretable in isolation — only the relative ranking between samples matters.

In [ ]:
import pickle

import haiku as hk
import jax
import jax.numpy as jnp
import pandas as pd

from multiomics_open_research.bulk_rna_bert.downstream.pretrained import get_pretrained_downstream_model
from multiomics_open_research.common.preprocess import preprocess_omic

In [ ]:
# Load data (identifier, survival_time, event are meta-columns; preprocess_omic drops them automatically)
rna_seq_df = pd.read_csv("../data/bulkrnabert/tcga_sample.csv")
print(f"{len(rna_seq_df)} samples")
rna_seq_df[["identifier", "survival_time", "event"]]

## 1. Cancer Type Classification

Predicts one of 5 TCGA cancer cohorts from RNA-seq expression. The model outputs logits over the 5 classes; `argmax` gives the predicted cohort.

In [ ]:
parameters, forward_fn, tokenizer, config, mlm_config = get_pretrained_downstream_model(
    model_name="tcga_5_cohorts",
    checkpoint_directory="../checkpoints/",
)
forward_fn = hk.transform(forward_fn)

In [ ]:
rna_seq_array = preprocess_omic(rna_seq_df, mlm_config)
tokens_ids = tokenizer.batch_tokenize(rna_seq_array)
tokens = jnp.asarray(tokens_ids, dtype=jnp.int32)

random_key = jax.random.PRNGKey(0)
outs = forward_fn.apply(parameters, random_key, tokens)

In [ ]:
with open("../data/bulkrnabert/5_cohorts_labels_mapping.pkl", "rb") as f:
    label_mapping = pickle.load(f)

results = pd.DataFrame({
    "identifier": rna_seq_df["identifier"],
    "predicted_cancer_type": [
        label_mapping[int(outs["logits"][i].argmax())]
        for i in range(len(rna_seq_df))
    ],
})
results

## 2. Pan-Cancer Survival

Predicts a **log partial hazard** (Cox linear predictor) for each sample.
- Higher score → higher predicted risk → shorter expected survival
- The score has no absolute meaning; only the **ranking between samples** is interpretable
- Proper evaluation (C-index, Kaplan-Meier) requires a large cohort; with 4 samples below we only illustrate whether the predicted ranking is consistent with observed survival times

In [ ]:
surv_parameters, surv_forward_fn, surv_tokenizer, surv_config, surv_mlm_config = get_pretrained_downstream_model(
    model_name="tcga_pancancer_survival",
    checkpoint_directory="../checkpoints/",
)
surv_forward_fn = hk.transform(surv_forward_fn)

In [ ]:
surv_array = preprocess_omic(rna_seq_df, surv_mlm_config)
surv_tokens_ids = surv_tokenizer.batch_tokenize(surv_array)
surv_tokens = jnp.asarray(surv_tokens_ids, dtype=jnp.int32)

surv_outs = surv_forward_fn.apply(surv_parameters, random_key, surv_tokens)
log_hazard_scores = surv_outs["logits"].squeeze(-1)

In [ ]:
import numpy as np

surv_results = pd.DataFrame({
    "identifier": rna_seq_df["identifier"],
    "observed_survival_days": rna_seq_df["survival_time"],
    "event": rna_seq_df["event"],
    "log_hazard_score": np.array(log_hazard_scores),
})

surv_results = surv_results.sort_values("log_hazard_score", ascending=False).reset_index(drop=True)
surv_results

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

colors = ["#C44E52" if i < 2 else "#4C72B0" for i in range(len(surv_results))]

# Left: predicted log-hazard scores
ax = axes[0]
ax.barh(surv_results["identifier"], surv_results["log_hazard_score"], color=colors, edgecolor="white")
ax.set_xlabel("Log partial hazard (higher = higher risk)")
ax.set_title("Predicted Risk Score")
ax.axvline(0, color="gray", linestyle="--", linewidth=0.8)

# Right: observed survival time
ax = axes[1]
ax.barh(surv_results["identifier"], surv_results["observed_survival_days"], color=colors, edgecolor="white")
ax.set_xlabel("Observed survival (days)")
ax.set_title("Actual Survival Time")
for i, row in surv_results.iterrows():
    ax.text(row["observed_survival_days"] + 10, i, f"{int(row['observed_survival_days'])}d", va="center", fontsize=9)

fig.suptitle(
    "Samples sorted by predicted risk (high→low) — higher risk (red) should correspond to shorter survival",
    fontsize=10, y=1.02,
)
plt.tight_layout()
plt.show()